# Locate Anything — Colab Live Demo

This notebook reuses the inference code already in our GitHub repo (`app.py`) to host a **public Gradio demo** of `nvidia/LocateAnything-3B` on Colab's free T4 GPU. The share link it produces is embedded on our GitHub Pages site.

**Before running anything:**
1. Go to **Runtime > Change runtime type > Hardware accelerator = T4 GPU**.
2. Run the cells top to bottom. The model-load cell takes a few minutes the first time (it downloads the 3B checkpoint).
3. The last cell prints a `https://xxxxx.gradio.live` link — copy it and paste it into `assets/config.js` on the website repo, then commit so GitHub Pages rebuilds.

> The share link is only valid while the launch cell is running. Colab kills idle sessions after ~90 min and caps sessions at ~12 h, so re-run the last two cells to refresh the link if needed.

In [ ]:
!nvidia-smi

In [ ]:
# Install dependencies, pinned to the same versions as the repo's requirements.txt
# where it's safe to do so. torch/torchvision stay as Colab's preinstalled CUDA
# builds, and numpy/Pillow are left at Colab's versions — downgrading them can
# break the Colab runtime. The model's custom modeling code imports lmdb and
# possibly cv2/peft at load time via transformers' check_imports, so those must
# be present before get_worker() runs or the import fails fast.
!pip install -q \
  "gradio==6.16.0" \
  "transformers==4.57.1" \
  "huggingface_hub==0.36.0" \
  accelerate \
  "lmdb==1.7.5" \
  "opencv-python-headless==4.11.0.86" \
  peft

# decord is only needed for video inputs; best-effort install (safe if the build fails).
!pip install -q decord==0.6.0 || echo "decord unavailable - video-only dependency, skipping"

# If you hit strange torch/CUDA errors with Colab's preinstalled torch, uncomment:
# !pip install -q "torch==2.8.0" "torchvision==0.23.0"

In [ ]:
import os
import pathlib
import sys

# Clone our repo so we can reuse app.py's LocateAnythingWorker + helpers verbatim.
REPO_URL = "https://github.com/isharmatech/Class-Project--Locate-Anything"
REPO_DIR = "/content/locate-anything"

if os.path.isdir(REPO_DIR):
    # Already cloned (e.g. re-running after a disconnect) — pull the latest instead.
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

app_py = pathlib.Path(REPO_DIR) / "app.py"
if not app_py.exists():
    raise FileNotFoundError(
        "app.py is missing from the cloned repo. Commit and push app.py "
        "(and requirements.txt) to GitHub first, then re-run this cell."
    )

sys.path.insert(0, REPO_DIR)
print("Repo ready:", app_py)

In [ ]:
# Reuse the repo's inference module — no rewriting of the model-loading code.
# get_worker() lazily loads LocateAnything-3B onto the T4 on first call.
# The model is public on Hugging Face, so no token is needed
# (app.py still honors the HF_TOKEN env var if one is set).
from app import build_demo, get_worker, MODEL_ID

print(f"Loading {MODEL_ID} onto the T4 (fp16 — T4 has no bf16) — first run downloads ~8 GB ...")
worker = get_worker()
print("Model loaded successfully.")

In [ ]:
# Build the same Gradio Blocks UI as our local app.py and expose a public share link.
# Copy the https://xxxxx.gradio.live URL printed below into assets/config.js.
demo = build_demo()
demo.queue().launch(share=True, debug=True)

## After the launch cell prints your link

1. Copy the `https://xxxxx.gradio.live` URL.
2. In the website repo, open `assets/config.js` and paste it into `GRADIO_LIVE_URL`.
3. Commit and push so GitHub Pages rebuilds with the live link.
4. Keep this Colab tab open and active for the whole presentation. If the session drops, re-run the **Load model** and **Launch** cells to get a fresh link and paste it again.

If the live link is offline on demo day, the website's `Try It Yourself` section automatically shows a fallback message and the recorded walkthrough video instead of a blank iframe.